# Customer Churn Prediction - Big Data Analytics Capstone 🚀

This notebook executes the entire HDFS → Hive → Spark pipeline from within Jupyter.

## Architecture
1. **HDFS**: Upload raw dataset.
2. **Hive**: Create database and managed/external tables.
3. **Spark**: Execute queries, clean data, perform EDA, and train ML Models.


## Step 1: Initialize PySpark Session
Configure Spark to use YARN and enable Hive support so we can run SQL commands directly on Hive.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Initialize Spark Session with Hive support
spark = SparkSession.builder     .appName("CustomerChurn_Jupyter_Pipeline")     .config("spark.sql.warehouse.dir", "/user/capstone/customer_churn/hive_warehouse")     .enableHiveSupport()     .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Version: {spark.version}")


## Step 2: HDFS Ingestion
We will create HDFS directories and upload our CSV dataset.
*Note: Make sure your  is in the  or  directory of your Jupyter container.*


In [ ]:
%%bash
# Setup HDFS directories
hdfs dfs -mkdir -p /user/capstone/customer_churn/raw
hdfs dfs -mkdir -p /user/capstone/customer_churn/processed
hdfs dfs -mkdir -p /user/capstone/customer_churn/results
hdfs dfs -mkdir -p /user/capstone/customer_churn/hive_warehouse

# Make sure the file exists locally in the Jupyter container before uploading
# Assuming you mapped the data folder to /home/jovyan/work/data or similar:
LOCAL_FILE="data/customer_churn_data.csv"
if [ -f "" ]; then
    echo "Uploading  to HDFS..."
    hdfs dfs -put -f  /user/capstone/customer_churn/raw/
else
    echo "Error:  not found in the Jupyter container."
    echo "Please adjust the path to point to your CSV file."
fi

# Verify HDFS Upload
hdfs dfs -ls /user/capstone/customer_churn/raw/


## Step 3: Hive Database & Table Creation (via Spark SQL)
We can execute Hive DDL directly through Spark SQL.


In [ ]:
# Create Database
spark.sql("CREATE DATABASE IF NOT EXISTS customer_churn_db LOCATION '/user/capstone/customer_churn/hive_warehouse'")
spark.sql("USE customer_churn_db")

# Create External Table (Raw Data)
spark.sql("""
CREATE EXTERNAL TABLE IF NOT EXISTS customer_churn_raw (
    customer_id         STRING,
    gender              STRING,
    senior_citizen      INT,
    partner             STRING,
    dependents          STRING,
    tenure_months       INT,
    phone_service       STRING,
    multiple_lines      STRING,
    internet_service    STRING,
    online_security     STRING,
    online_backup       STRING,
    device_protection   STRING,
    tech_support        STRING,
    streaming_tv        STRING,
    streaming_movies    STRING,
    contract            STRING,
    paperless_billing   STRING,
    payment_method      STRING,
    monthly_charges     DOUBLE,
    total_charges       DOUBLE,
    churn               STRING
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
STORED AS TEXTFILE
LOCATION '/user/capstone/customer_churn/raw'
TBLPROPERTIES ('skip.header.line.count'='1')
""")

# Show tables
spark.sql("SHOW TABLES").show()


### Create Managed Processed Table


In [ ]:
spark.sql("DROP TABLE IF EXISTS customer_churn_processed")

spark.sql("""
CREATE TABLE customer_churn_processed (
    customer_id         STRING,
    gender              STRING,
    senior_citizen      INT,
    partner             STRING,
    dependents          STRING,
    tenure_months       INT,
    phone_service       STRING,
    multiple_lines      STRING,
    internet_service    STRING,
    online_security     STRING,
    online_backup       STRING,
    device_protection   STRING,
    tech_support        STRING,
    streaming_tv        STRING,
    streaming_movies    STRING,
    contract            STRING,
    paperless_billing   STRING,
    payment_method      STRING,
    monthly_charges     DOUBLE,
    total_charges       DOUBLE,
    churn               STRING,
    tenure_group        STRING
) STORED AS ORC
""")

# Populate Processed table with groupings
spark.sql("""
INSERT OVERWRITE TABLE customer_churn_processed
SELECT
    customer_id, gender, senior_citizen, partner, dependents,
    tenure_months, phone_service, multiple_lines, internet_service, online_security,
    online_backup, device_protection, tech_support, streaming_tv, streaming_movies,
    contract, paperless_billing, payment_method, monthly_charges,
    total_charges, churn,
    CASE
        WHEN tenure_months <= 12 THEN '0-12 months'
        WHEN tenure_months <= 24 THEN '13-24 months'
        WHEN tenure_months <= 48 THEN '25-48 months'
        ELSE '49-72 months'
    END AS tenure_group
FROM customer_churn_raw
WHERE customer_id IS NOT NULL AND customer_id != 'customer_id'
""")

print("Data loaded into managed Hive table.")
spark.sql("SELECT COUNT(*) as Processed_Count FROM customer_churn_processed").show()


## Step 4: Hive Analytics Queries
We can run our analytical queries against the processed Hive table.


In [ ]:
print("1. Overall Churn Rate")
spark.sql("""
SELECT churn, COUNT(*) AS count, 
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
FROM customer_churn_processed GROUP BY churn
""").show()

print("2. Churn by Contract Type")
spark.sql("""
SELECT contract, COUNT(*) AS total, 
       SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
       ROUND(SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS churn_rate
FROM customer_churn_processed GROUP BY contract ORDER BY churn_rate DESC
""").show()


## Step 5: Spark Data Transformation & ML Pipeline
Now we'll load the data directly from Hive as a DataFrame and prepare it for Machine Learning.


In [ ]:
# Read data from Hive
df_clean = spark.table("customer_churn_processed")

# Add Feature columns (total services, etc)
df_clean = df_clean     .withColumn("total_services",
        F.when(F.col("phone_service") == "Yes", 1).otherwise(0) +
        F.when(F.col("multiple_lines") == "Yes", 1).otherwise(0) +
        F.when(F.col("internet_service") != "No", 1).otherwise(0) +
        F.when(F.col("online_security") == "Yes", 1).otherwise(0) +
        F.when(F.col("online_backup") == "Yes", 1).otherwise(0) +
        F.when(F.col("device_protection") == "Yes", 1).otherwise(0) +
        F.when(F.col("tech_support") == "Yes", 1).otherwise(0) +
        F.when(F.col("streaming_tv") == "Yes", 1).otherwise(0) +
        F.when(F.col("streaming_movies") == "Yes", 1).otherwise(0)
    )     .withColumn("churn_label", F.when(F.col("churn") == "Yes", 1).otherwise(0))

df_clean.cache()
df_clean.show(5)


### ML Pipeline Setup


In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline

categorical_cols = [
    "gender", "partner", "dependents", "phone_service", "multiple_lines",
    "internet_service", "online_security", "online_backup", "device_protection",
    "tech_support", "streaming_tv", "streaming_movies",
    "contract", "paperless_billing", "payment_method", "tenure_group"
]

numerical_cols = ["senior_citizen", "tenure_months", "monthly_charges", "total_charges", "total_services"]

indexers = [StringIndexer(inputCol=col, outputCol=col + "_idx", handleInvalid="keep") for col in categorical_cols]
encoders = [OneHotEncoder(inputCol=col + "_idx", outputCol=col + "_vec") for col in categorical_cols]

feature_cols = [col + "_vec" for col in categorical_cols] + numerical_cols
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw", handleInvalid="keep")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=False)

# Train/Test Split
train_df, test_df = df_clean.randomSplit([0.8, 0.2], seed=42)


### Train Random Forest Model


In [ ]:
rf = RandomForestClassifier(featuresCol="features", labelCol="churn_label", numTrees=100, maxDepth=8, seed=42)
rf_pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, rf])

print("Training Random Forest model...")
rf_model = rf_pipeline.fit(train_df)
predictions = rf_model.transform(test_df)

evaluator = BinaryClassificationEvaluator(labelCol="churn_label", metricName="areaUnderROC")
auc = evaluator.evaluate(predictions)
print(f"Random Forest AUC-ROC: {auc:.4f}")


## Step 6: Save Results to HDFS


In [ ]:
predictions.select("customer_id", "churn", "churn_label", "prediction", "probability")     .write.mode("overwrite")     .parquet("hdfs:///user/capstone/customer_churn/results/churn_predictions.parquet")

print("Predictions saved successfully to HDFS!")
